In [ ]:
import pandas as pd
import numpy as np  
from sklearn.preprocessing import RobustScaler

name = input("Ведите имя файла большими буквами: ")

In [ ]:
df = pd.read_csv(f"../df/clean_df/{name}.csv")
df

In [ ]:
scale_cols = [
    "Open", "High", "Low", "Close",
    "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
    "AO",
    "AddOn_Anchor_Level", "AddOn_Size_Pct"
]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])


In [ ]:
columns = [
    "AddOn_Anchor_Level",
    "AddOn_Anchor_IsUp",
    "AddOn_Size_Pct"
]

df = df.dropna(subset=columns).reset_index(drop=True)

In [ ]:
df

In [ ]:
def build_target(df: pd.DataFrame, h: int = 20, target_type: str = "classification") -> pd.DataFrame:

    df = df.copy()

    df["Close_fwd"] = df["Close"].shift(-h)

    df["ret_H"] = np.where(
        df["EntrySignal"] > 0,
        (df["Close_fwd"] - df["Close"]) / df["Close"],

        np.where(
            df["EntrySignal"] < 0,
            (df["Close"] - df["Close_fwd"]) / df["Close"],
            0.0
        )
    )

    if target_type == "classification":
        df["GoodTrade"] = ((df["EntrySignal"] != 0) & (df["ret_H"] > 0)).astype(int)

    df = df.dropna(subset=["Close_fwd"])

    return df

In [ ]:
df = build_target(df, h=20)